# Data Loading, Storage, 

In [ ]:
import sys
!conda install --yes --prefix {sys.prefix} -c conda-forge yfinance
#!{sys.executable} -m pip install yfinance

In [ ]:
import numpy as np
import pandas as pd

## Reading and Writing Data in Text Format

Handling dates and other custom types can require extra effort. Let’s start with a small comma-separated (CSV) text file

Here I used the Unix cat shell command to print the raw contents of the file to the screen. If you’re on Windows, you can use type instead of cat to achieve the same effect. Since this is comma-delimited, we can use read_csv to read it into a DataFrame:

In [7]:
df = pd.read_csv('examples-20240404/examples/ex1.csv')
df

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


We could also have used read_table and specified the delimiter

In [9]:
pd.read_table('examples-20240404/examples/ex1.csv', sep=',')

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


A file will not always have a header row. Consider this file

In [10]:
!cat examples-20240404/examples/ex1.csv

'cat' is not recognized as an internal or external command,
operable program or batch file.


To read this file, you have a couple of options. You can allow pandas to assign default column names, or you can specify names yourself

In [11]:
pd.read_csv('examples-20240404/examples/ex2.csv', header=None)

,0,1,2,3,4
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


In [12]:
pd.read_csv('examples-20240404/examples/ex2.csv', names=['a', 'b', 'c', 'd', 'message'])

,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Suppose you wanted the message column to be the index of the returned DataFrame. You can either indicate you want the column at index 4 or named 'message' using the index_col argument

In [13]:
names = ['a', 'b', 'c', 'd', 'message']
pd.read_csv('examples-20240404/examples/ex2.csv', names=names, index_col='message')

,a,b,c,d
message,,,,
hello,1,2,3,4
world,5,6,7,8
foo,9,10,11,12


In the event that you want to form a hierarchical index from multiple columns, pass a list of column numbers or names

In [ ]:
!cat data/examples/csv_mindex.csv

In [16]:
parsed = pd.read_csv('examples-20240404/examples/csv_mindex.csv',
                     index_col=['key1', 'key2'])
parsed

value1  value2
key1 key2                
one  a          1       2
     b          3       4
     c          5       6
     d          7       8
two  a          9      10
     b         11      12
     c         13      14
     d         15      16

In some cases, a table might not have a fixed delimiter, using whitespace or some other pattern to separate fields. Consider a text file that looks like this

In [17]:
list(open('examples-20240404/examples/ex3.txt'))

['            A         B         C\n',
 'aaa -0.264438 -1.026059 -0.619500\n',
 'bbb  0.927272  0.302904 -0.032399\n',
 'ccc -0.264273 -0.386314 -0.217601\n',
 'ddd -0.871858 -0.348382  1.100491\n']

While you could do some munging by hand, the fields here are separated by a variable amount of whitespace. In these cases, you can pass a regular expression as a delimiter for read_table. This can be expressed by the regular expression \s+, so we have then

In [18]:
result = pd.read_table('examples-20240404/examples/ex3.txt', sep='\s+')
result

,A,B,C
aaa,-0.264438,-1.026059,-0.619500
bbb,0.927272,0.302904,-0.032399
ccc,-0.264273,-0.386314,-0.217601
ddd,-0.871858,-0.348382,1.100491


Because there was one fewer column name than the number of data rows, read_table infers that the first column should be the DataFrame’s index in this special case.
The parser functions have many additional arguments to help you handle the wide variety of exception file formats that occur (see a partial listing in Table 6-2). For example, you can skip the first, third, and fourth rows of a file with skiprows

In [19]:
!cat data/examples/ex4.csv
pd.read_csv('examples-20240404/examples/ex4.csv', skiprows=[0, 2, 3])

'cat' is not recognized as an internal or external command,
operable program or batch file.


,a,b,c,d,message
0,1,2,3,4,hello
1,5,6,7,8,world
2,9,10,11,12,foo


Handling missing values is an important and frequently nuanced part of the file parsing process. Missing data is usually either not present (empty string) or marked by some sentinel value. By default, pandas uses a set of commonly occurring sentinels, such as NA and NULL

In [20]:
!cat data/examples/ex5.csv
result = pd.read_csv('examples-20240404/examples/ex5.csv')
result

'cat' is not recognized as an internal or external command,
operable program or batch file.


,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


In [21]:
pd.isnull(result)

,something,a,b,c,d,message
0,False,False,False,False,False,True
1,False,False,False,True,False,False
2,False,False,False,False,False,False


The na_values option can take either a list or set of strings to consider missing values

In [22]:
result = pd.read_csv('examples-20240404/examples/ex5.csv', na_values=['NULL'])
result

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


Different NA sentinels can be specified for each column in a dict

In [23]:
sentinels = {'message': ['foo', 'NA'], 'something': ['two']}
pd.read_csv('examples-20240404/examples/ex5.csv', na_values=sentinels)

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,NaN,5,6,NaN,8,world
2,three,9,10,11.0,12,NaN


### Reading Text Files in Pieces

When processing very large files or figuring out the right set of arguments to correctly process a large file, you may only want to read in a small piece of a file or iterate through smaller chunks of the file.
Before we look at a large file, we make the pandas display settings more compact

In [27]:
pd.set_option('display.max_columns', 6)
pd.set_option('display.max_rows', 200) ### if we have 1000 and we ask for 200 we will only see five rows unless we set max_rows = 1000
pd.set_option('display.min_rows', 5) ### Will always display this number
pd.set_option('display.expand_frame_repr', True)

In [28]:
pd.set_option('display.min_rows', 10)

In [29]:
result = pd.read_csv('examples-20240404/examples/ex6.csv')
result

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


If you want to only read a small number of rows (avoiding reading the entire file), specify that with nrows

In [30]:
pd.read_csv('examples-20240404/examples/ex6.csv', nrows=10)

,one,two,three,four,key
0,0.467976,-0.038649,-0.295344,-1.824726,L
1,-0.358893,1.404453,0.704965,-0.200638,B
2,-0.501840,0.659254,-0.421691,-0.057688,G
3,0.204886,1.074134,1.388361,-0.982404,R
4,0.354628,-0.133116,0.283763,-0.837063,Q
5,1.817480,0.742273,0.419395,-2.251035,Q
6,-0.776764,0.935518,-0.332872,-1.875641,U
7,-0.913135,1.530624,-0.572657,0.477252,K
8,0.358480,-0.497572,-0.367016,0.507702,S
9,-1.740877,-1.160417,-1.637830,2.172201,G


To read a file in pieces, specify a chunksize as a number of rows

In [31]:
chunker = pd.read_csv('examples-20240404/examples/ex6.csv', chunksize=1000)
chunker

The TextParser object returned by read_csv allows you to iterate over the parts of the file according to the chunksize. For example, we can iterate over ex6.csv, aggregating the value counts in the 'key' column like so

In [36]:
chunker = pd.read_csv('examples-20240404/examples/ex6.csv', chunksize=1000)

tot = pd.Series([])
for piece in chunker:
    tot = tot.add(piece['key'].value_counts(), fill_value=0)

tot = tot.sort_values(ascending=False)

In [35]:
piece

,one,two,three,four,key
9000,0.467976,-0.038649,-0.295344,-1.824726,B
9001,-0.358893,1.404453,0.704965,-0.200638,M
9002,-0.501840,0.659254,-0.421691,-0.057688,N
9003,0.204886,1.074134,1.388361,-0.982404,N
9004,0.354628,-0.133116,0.283763,-0.837063,Y
...,...,...,...,...,...
9995,2.311896,-0.417070,-1.409599,-0.515821,L
9996,-0.479893,-0.650419,0.745152,-0.646038,E
9997,0.523331,0.787112,0.486066,1.093156,K
9998,-0.362559,0.598894,-1.843201,0.887292,G


In [37]:
tot

key
E    368
X    364
L    346
O    343
Q    340
M    338
J    337
F    335
K    334
H    330
V    328
I    327
U    326
P    324
D    320
A    320
R    318
Y    314
G    308
S    308
N    306
W    305
T    304
B    302
Z    288
C    286
4    171
6    166
7    164
8    162
3    162
5    157
2    152
0    151
9    150
1    146
dtype: object

### Writing Data to Text Format

Data can also be exported to a delimited format. Let’s consider one of the CSV files read before

In [38]:
data = pd.read_csv('examples-20240404/examples/ex5.csv')
data

,something,a,b,c,d,message
0,one,1,2,3.0,4,NaN
1,two,5,6,NaN,8,world
2,three,9,10,11.0,12,foo


Using DataFrame’s to_csv method, we can write the data out to a comma-separated file

In [39]:
data.to_csv('examples-20240404/examples/out_2.csv')
!cat data/examples/out.csv

'cat' is not recognized as an internal or external command,
operable program or batch file.


Other delimiters can be used, of course (writing to sys.stdout so it prints the text result to the console)

In [40]:
import sys
data.to_csv(sys.stdout, sep='|')

|something|a|b|c|d|message
0|one|1|2|3.0|4|
1|two|5|6||8|world
2|three|9|10|11.0|12|foo


Missing values appear as empty strings in the output. You might want to denote them by some other sentinel value

In [41]:
data.to_csv(sys.stdout, na_rep='NULL')

,something,a,b,c,d,message
0,one,1,2,3.0,4,NULL
1,two,5,6,NULL,8,world
2,three,9,10,11.0,12,foo


With no other options specified, both the row and column labels are written. Both of these can be disabled

In [42]:
data.to_csv(sys.stdout, index=False, header=False)

one,1,2,3.0,4,
two,5,6,,8,world
three,9,10,11.0,12,foo


You can also write only a subset of the columns, and in an order of your choosing:

In [43]:
data.to_csv(sys.stdout, index=False, columns=['a', 'b', 'c'])

a,b,c
1,2,3.0
5,6,
9,10,11.0


Series also has a to_csv method

In [44]:
dates = pd.date_range('1/1/2000', periods=7)
ts = pd.Series(np.arange(7), index=dates)
ts

2000-01-01    0
2000-01-02    1
2000-01-03    2
2000-01-04    3
2000-01-05    4
2000-01-06    5
2000-01-07    6
Freq: D, dtype: int32

In [45]:
dates = pd.date_range('1/1/2000', periods=7)
ts = pd.Series(np.arange(7), index=dates)
ts.to_csv('examples-20240404/examples/tseries.csv')
!cat data/examples/tseries.csv

'cat' is not recognized as an internal or external command,
operable program or batch file.


### Working with Delimited Formats

It’s possible to load most forms of tabular data from disk using functions like pandas.read_table. In some cases, however, some manual processing may be necessary. It’s not uncommon to receive a file with one or more malformed lines that trip up read_table. To illustrate the basic tools, consider a small CSV file

In [ ]:
!cat data/examples/ex7.csv

For any file with a single-character delimiter, you can use Python’s built-in csv module. To use it, pass any open file or file-like object to csv.reader

In [46]:
import csv
f = open('examples-20240404/examples/ex7.csv')

reader = csv.reader(f)
reader

Iterating through the reader like a file yields tuples of values with any quote characters removed:

In [47]:
for line in reader:
    print(line)

['a', 'b', 'c']
['1', '2', '3']
['1', '2', '3']


From there, it’s up to you to do the wrangling necessary to put the data in the form that you need it. Let’s take this step by step. First, we read the file into a list of lines

In [48]:
with open('examples-20240404/examples/ex7.csv') as f:
    lines = list(csv.reader(f))

Then, we split the lines into the header line and the data lines:

In [49]:
header, values = lines[0], lines[1:]

In [50]:
lines[0]

['a', 'b', 'c']

### JSON Data

JSON (short for JavaScript Object Notation) has become one of the standard formats for sending data by HTTP request between web browsers and other applications. It is a much more free-form data format than a tabular text form like CSV. Here is an example

In [51]:
obj = """
{"name": "Wes",
 "places_lived": ["United States", "Spain", "Germany"],
 "pet": null,
 "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]},
              {"name": "Katie", "age": 38,
               "pets": ["Sixes", "Stache", "Cisco"]}]
}
"""

JSON is very nearly valid Python code with the exception of its null value null and some other nuances (such as disallowing trailing commas at the end of lists). The basic types are objects (dicts), arrays (lists), strings, numbers, booleans, and nulls. All of the keys in an object must be strings. There are several Python libraries for reading
and writing JSON data. I’ll use json here, as it is built into the Python standard library. To convert a JSON string to Python form, use json.loads

In [52]:
import json
result = json.loads(obj)
result

{'name': 'Wes',
 'places_lived': ['United States', 'Spain', 'Germany'],
 'pet': None,
 'siblings': [{'name': 'Scott', 'age': 30, 'pets': ['Zeus', 'Zuko']},
  {'name': 'Katie', 'age': 38, 'pets': ['Sixes', 'Stache', 'Cisco']}]}

In [53]:
result['siblings'][0]['age']

30

json.dumps, on the other hand, converts a Python object back to JSON

In [54]:
asjson = json.dumps(result)

In [55]:
asjson

'{"name": "Wes", "places_lived": ["United States", "Spain", "Germany"], "pet": null, "siblings": [{"name": "Scott", "age": 30, "pets": ["Zeus", "Zuko"]}, {"name": "Katie", "age": 38, "pets": ["Sixes", "Stache", "Cisco"]}]}'

How you convert a JSON object or list of objects to a DataFrame or some other data structure for analysis will be up to you. Conveniently, you can pass a list of dicts (which were previously JSON objects) to the DataFrame constructor and select a subset of the data fields

In [56]:
siblings = pd.DataFrame(result['siblings'], columns=['name', 'age'])
siblings

,name,age
0,Scott,30
1,Katie,38


The pandas.read_json can automatically convert JSON datasets in specific arrangements into a Series or DataFrame. For example

In [57]:
!cat data/examples/example.json

'cat' is not recognized as an internal or external command,
operable program or batch file.


The default options for pandas.read_json assume that each object in the JSON array is a row in the table:

In [59]:
data = pd.read_json('examples-20240404/examples/example.json')
data

,a,b,c
0,1,2,3
1,4,5,6
2,7,8,9


For an extended example of reading and manipulating JSON data (including nested records), see the USDA Food Database example in Chapter 7.
If you need to export data from pandas to JSON, one way is to use the to_json meth‐ ods on Series and DataFrame

In [60]:
print(data.to_json())
print(data.to_json(orient='records'))

{"a":{"0":1,"1":4,"2":7},"b":{"0":2,"1":5,"2":8},"c":{"0":3,"1":6,"2":9}}
[{"a":1,"b":2,"c":3},{"a":4,"b":5,"c":6},{"a":7,"b":8,"c":9}]


### Reading Microsoft Excel Files

pandas also supports reading tabular data stored in Excel 2003 (and higher) files using either the ExcelFile class or pandas.read_excel function. Internally these tools use the add-on packages xlrd and openpyxl to read XLS and XLSX files, respectively. You may need to install these manually with pip or conda.
To use ExcelFile, create an instance by passing a path to an xls or xlsx file:

In [61]:
xlsx = pd.ExcelFile('examples-20240404/examples/ex1.xlsx')

In [62]:
openpyxl

NameError: name 'openpyxl' is not defined

Data stored in a sheet can then be read into DataFrame with parse:

In [69]:
import openpyxl
pd.read_excel(xlsx, 'Sheet1')

,Unnamed: 0,a,b,c,d,message
0,0,1,2,3,4,hello
1,1,5,6,7,8,world
2,2,9,10,11,12,foo


If you are reading multiple sheets in a file, then it is faster to create the ExcelFile, but you can also simply pass the filename to pandas.read_excel

In [70]:
frame = pd.read_excel('examples-20240404/examples/ex1.xlsx', 'Sheet1')
frame

,Unnamed: 0,a,b,c,d,message
0,0,1,2,3,4,hello
1,1,5,6,7,8,world
2,2,9,10,11,12,foo


To write pandas data to Excel format, you must first create an ExcelWriter, then write data to it using pandas objects’ to_excel method:

In [71]:
writer = pd.ExcelWriter('examples-20240404/examples/ex2.xlsx')
frame.to_excel(writer, 'Sheet1')
writer.save()

AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

You can also pass a file path to to_excel and avoid the ExcelWriter:

In [67]:
frame.to_excel('examples-20240404/examples/ex2.xlsx')

In [68]:
!rm examples-20240404/examples/ex2.xlsx

'rm' is not recognized as an internal or external command,
operable program or batch file.


## Interacting with Web APIs

Many websites have public APIs providing data feeds via JSON or some other format. There are a number of ways to access these APIs from Python; one easy-to-use method that I recommend is the requests package. To find the last 30 GitHub issues for pandas on GitHub, we can make a GET HTTP request using the add-on requests library.

In [72]:
import requests
url = 'https://api.github.com/repos/pandas-dev/pandas/issues'
resp = requests.get(url)
resp

<Response [200]>

The Response object’s json method will return a dictionary containing JSON parsed into native Python objects

In [73]:
data = resp.json()
data

[{'url': 'https://api.github.com/repos/pandas-dev/pandas/issues/58149',
  'repository_url': 'https://api.github.com/repos/pandas-dev/pandas',
  'labels_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/58149/labels{/name}',
  'comments_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/58149/comments',
  'events_url': 'https://api.github.com/repos/pandas-dev/pandas/issues/58149/events',
  'html_url': 'https://github.com/pandas-dev/pandas/issues/58149',
  'id': 2225513739,
  'node_id': 'I_kwDOAA0YD86EpqUL',
  'number': 58149,
  'title': 'BUG: Pandas does not recognise older missing value code for double when reading Stata files prior to 108 (Stata 6) format',
  'user': {'login': 'cmjcharlton',
   'id': 90400333,
   'node_id': 'MDQ6VXNlcjkwNDAwMzMz',
   'avatar_url': 'https://avatars.githubusercontent.com/u/90400333?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/cmjcharlton',
   'html_url': 'https://github.com/cmjcharlton',
   'followers_url': 'h

In [74]:
data[0]['url']

'https://api.github.com/repos/pandas-dev/pandas/issues/58149'

Each element in data is a dictionary containing all of the data found on a GitHub issue page (except for the comments). We can pass data directly to DataFrame and extract fields of interest:

In [75]:
issues = pd.DataFrame(data, columns=['number', 'title',
                                     'labels', 'state'])
issues

,number,title,labels,state
0,58149,BUG: Pandas does not recognise older missing v...,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
1,58148,BUG: Fix Index.sort_values with natsort_key ra...,[],open
2,58147,BUG: Casting dtype of large datetime values ra...,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
3,58146,BUG: incorrect aggregation of dataframe when u...,"[{'id': 76811, 'node_id': 'MDU6TGFiZWw3NjgxMQ=...",open
4,58145,REGR: from_records not initializing subclasses...,"[{'id': 32815646, 'node_id': 'MDU6TGFiZWwzMjgx...",open
5,58143,REF: Allow Index._with_infer to also return Ra...,"[{'id': 1218227310, 'node_id': 'MDU6TGFiZWwxMj...",open
6,58142,QST: loc returns matrix with one row when ind...,"[{'id': 34444536, 'node_id': 'MDU6TGFiZWwzNDQ0...",open
7,58141,ENH: Consistent naming conventions for string ...,"[{'id': 76812, 'node_id': 'MDU6TGFiZWw3NjgxMg=...",open
8,58139,BUG: fixed to_numeric loss in precision when c...,[],open
9,58136,ENH: Processing of .mask() for pd.NA #56844,"[{'id': 2822342, 'node_id': 'MDU6TGFiZWwyODIyM...",open


With a bit of elbow grease, you can create some higher-level interfaces to common web APIs that return DataFrame objects for easy analysis.

### YAHOO FINANCE

In [76]:
import sys
!conda install --yes --prefix {sys.prefix} -c conda-forge yfinance
#!{sys.executable} -m pip install openpyxl 

Solving environment: ...working... done

# All requested packages already installed.





==> WARNING: A newer version of conda exists. <==
  current version: 23.7.4
  latest version: 24.3.0

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=24.3.0




In [77]:
conda list

# packages in environment at C:\Users\karol\anaconda3:
#
# Name                    Version                   Build  Channel
_anaconda_depends         2023.09             py311_mkl_1  
abseil-cpp                20211102.0           hd77b12b_0  
aiobotocore               2.5.0           py311haa95532_0  
aiofiles                  22.1.0          py311haa95532_0  
aiohttp                   3.8.5           py311h2bbff1b_0  
aioitertools              0.7.1              pyhd3eb1b0_0  
aiosignal                 1.2.0              pyhd3eb1b0_0  
aiosqlite                 0.18.0          py311haa95532_0  
alabaster                 0.7.12             pyhd3eb1b0_0  
anaconda-anon-usage       0.4.2           py311hfc23b7f_0  
anaconda-catalogs         0.2.0           py311haa95532_0  
anaconda-client           1.12.1          py311haa95532_0  
anaconda-cloud-auth       0.1.3           py311haa95532_0  
anaconda-navigator        2.5.0           py311haa95532_0  
anaconda-project          0.11.1    

In [78]:
import yfinance as yf
msft = yf.Ticker("MSFT")
print(msft)

yfinance.Ticker object <MSFT>


In [79]:
msft.info

{'address1': 'One Microsoft Way',
 'city': 'Redmond',
 'state': 'WA',
 'zip': '98052-6399',
 'country': 'United States',
 'phone': '425 882 8080',
 'website': 'https://www.microsoft.com',
 'industry': 'Software - Infrastructure',
 'industryKey': 'software-infrastructure',
 'industryDisp': 'Software - Infrastructure',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Microsoft Corporation develops and supports software, services, devices and solutions worldwide. The Productivity and Business Processes segment offers office, exchange, SharePoint, Microsoft Teams, office 365 Security and Compliance, Microsoft viva, and Microsoft 365 copilot; and office consumer services, such as Microsoft 365 consumer subscriptions, Office licensed on-premises, and other office services. This segment also provides LinkedIn; and dynamics business solutions, including Dynamics 365, a set of intelligent, cloud-based applications across ERP, CRM, power 

In [80]:
msft.dividends

Date
2003-02-19 00:00:00-05:00    0.08
2003-10-15 00:00:00-04:00    0.16
2004-08-23 00:00:00-04:00    0.08
2004-11-15 00:00:00-05:00    3.08
2005-02-15 00:00:00-05:00    0.08
2005-05-16 00:00:00-04:00    0.08
2005-08-15 00:00:00-04:00    0.08
2005-11-15 00:00:00-05:00    0.08
2006-02-15 00:00:00-05:00    0.09
2006-05-15 00:00:00-04:00    0.09
2006-08-15 00:00:00-04:00    0.09
2006-11-14 00:00:00-05:00    0.10
2007-02-13 00:00:00-05:00    0.10
2007-05-15 00:00:00-04:00    0.10
2007-08-14 00:00:00-04:00    0.10
2007-11-13 00:00:00-05:00    0.11
2008-02-19 00:00:00-05:00    0.11
2008-05-13 00:00:00-04:00    0.11
2008-08-19 00:00:00-04:00    0.11
2008-11-18 00:00:00-05:00    0.13
2009-02-17 00:00:00-05:00    0.13
2009-05-19 00:00:00-04:00    0.13
2009-08-18 00:00:00-04:00    0.13
2009-11-17 00:00:00-05:00    0.13
2010-02-16 00:00:00-05:00    0.13
2010-05-18 00:00:00-04:00    0.13
2010-08-17 00:00:00-04:00    0.13
2010-11-16 00:00:00-05:00    0.16
2011-02-15 00:00:00-05:00    0.16
2011-05-1

In [81]:
msft.history(period="1mo")

,Open,High,Low,...,Volume,Dividends,Stock Splits
Date,,,,,,,
2024-03-04 00:00:00-05:00,413.440002,417.350006,412.320007,...,17596000,0.0,0.0
2024-03-05 00:00:00-05:00,413.959991,414.250000,400.640015,...,26919200,0.0,0.0
2024-03-06 00:00:00-05:00,402.970001,405.160004,398.390015,...,22344100,0.0,0.0
2024-03-07 00:00:00-05:00,406.119995,409.779999,402.239990,...,18718500,0.0,0.0
2024-03-08 00:00:00-05:00,407.959991,410.420013,404.329987,...,17971700,0.0,0.0
2024-03-11 00:00:00-04:00,403.760010,405.679993,401.260010,...,16120800,0.0,0.0
2024-03-12 00:00:00-04:00,407.619995,415.570007,406.790009,...,22457000,0.0,0.0
2024-03-13 00:00:00-04:00,418.100006,418.179993,411.450012,...,17115900,0.0,0.0
2024-03-14 00:00:00-04:00,420.239990,427.820007,417.989990,...,34157300,0.0,0.0


You can also download data for multiple tickers at once, like before

In [82]:
data = yf.download("SPY AAPL", start="2017-01-01", end="2017-04-30")
data

[*********************100%%**********************]  2 of 2 completed


Price       Adj Close                  Close  ...        Open     Volume  \
Ticker           AAPL         SPY       AAPL  ...         SPY       AAPL   
Date                                          ...                          
2017-01-03  26.989264  199.200638  29.037500  ...  225.039993  115127600   
2017-01-04  26.959057  200.385696  29.004999  ...  225.619995   84472400   
2017-01-05  27.096155  200.226471  29.152500  ...  226.270004   88774400   
2017-01-06  27.398233  200.942886  29.477501  ...  226.529999  127007600   
2017-01-09  27.649185  200.279541  29.747499  ...  226.910004  134247600   
2017-01-10  27.677073  200.279541  29.777500  ...  226.479996   97848400   
2017-01-11  27.825779  200.845596  29.937500  ...  226.360001  110354400   
2017-01-12  27.709599  200.341461  29.812500  ...  226.500000  108344800   
2017-01-13  27.660805  200.801346  29.760000  ...  226.729996  104447600   
2017-01-17  27.883873  200.093872  30.000000  ...  226.309998  137759200   
2017-01-18  27.881550  200.536026  29.997499  ...  226.539993   94852000   
2017-01-19  27.832754  199.793182  29.945000  ...  226.839996  102389200   
2017-01-20  27.883873  200.527222  30.000000  ...  226.699997  130391600   
2017-01-23  27.902464  200.005447  30.020000  ...  226.740005   88200800   
2017-01-24  27.876907  201.287781  29.992500  ...  226.399994   92844000   
2017-01-25  28.320717  203.030029  30.469999  ...  228.699997  129510400   
2017-01-26  28.334661  202.817810  30.485001  ...  229.399994  105350400   
2017-01-27  28.336987  202.499435  30.487499  ...  229.419998   82251600   
2017-01-30  28.262629  201.243561  30.407499  ...  228.169998  121510000   
2017-01-31  28.197565  201.225876  30.337500  ...  226.979996  196804000   
2017-02-01  29.917067  201.305481  32.187500  ...  227.529999  447940000   
2017-02-02  29.865955  201.438126  32.132500  ...  227.619995  134841600   
2017-02-03  29.993752  202.826660  32.270000  ...  228.820007   98029200   
2017-02-06  30.274923  202.463989  32.572498  ...  228.869995  107383600   
2017-02-07  30.563047  202.472824  32.882500  ...  229.380005  152735200   
2017-02-08  30.681547  202.738190  33.009998  ...  228.940002   92016400   
2017-02-09  30.903255  203.940948  33.105000  ...  229.240005  113399600   
2017-02-10  30.833244  204.745728  33.029999  ...  231.000000   80262000   
2017-02-13  31.106287  205.860092  33.322498  ...  232.080002   92141600   
2017-02-14  31.510036  206.682587  33.755001  ...  232.559998  132904800   
2017-02-15  31.624384  207.761551  33.877499  ...  233.449997  142492400   
2017-02-16  31.587055  207.584656  33.837502  ...  234.949997   90338400   
2017-02-17  31.673388  207.911880  33.930000  ...  233.949997   88792800   
2017-02-21  31.902086  209.150055  34.174999  ...  235.520004   98028800   
2017-02-22  31.997780  208.964264  34.277500  ...  236.020004   83347600   
2017-02-23  31.862427  209.105835  34.132500  ...  236.880005   83152800   
2017-02-24  31.892761  209.371140  34.165001  ...  235.460007   87106400   
2017-02-27  31.955772  209.698380  34.232498  ...  236.639999   81029600   
2017-02-28  31.969770  209.132294  34.247501  ...  236.669998   93931600   
2017-03-01  32.623222  212.059708  34.947498  ...  238.389999  145658400   
2017-03-02  32.429520  210.724243  34.740002  ...  239.559998  104844000   
2017-03-03  32.620884  210.856918  34.945000  ...  238.169998   84432400   
2017-03-06  32.518192  210.228973  34.834999  ...  237.500000   87000000   
2017-03-07  32.560211  209.601089  34.880001  ...  237.360001   69785200   
2017-03-08  32.438850  209.211960  34.750000  ...  237.339996   74828800   
2017-03-09  32.364178  209.477234  34.669998  ...  236.699997   88623600   
2017-03-10  32.471527  210.211258  34.785000  ...  237.970001   78451200   
2017-03-13  32.485531  210.317398  34.799999  ...  237.619995   69686800   
2017-03-14  32.436527  209.512619  34.747501  ...  237.179993   61236400   
2017-03-15  32.779583  211.325607  35.115002  ..

In [83]:
msft = yf.Ticker("MSFT")
msft.balance_sheet

,2023-06-30,2022-06-30,2021-06-30,2020-06-30
Ordinary Shares Number,7432000000.0,7464000000.0,7519000000.0,7571000000.0
Share Issued,7432000000.0,7464000000.0,7519000000.0,7571000000.0
Net Debt,12533000000.0,35850000000.0,43922000000.0,49751000000.0
Total Debt,59965000000.0,61270000000.0,67775000000.0,70998000000.0
Tangible Book Value,128971000000.0,87720000000.0,84477000000.0,67915000000.0
Invested Capital,253460000000.0,216323000000.0,200134000000.0,181631000000.0
Working Capital,80108000000.0,74602000000.0,95749000000.0,109605000000.0
Net Tangible Assets,128971000000.0,87720000000.0,84477000000.0,67915000000.0
Capital Lease Obligations,12728000000.0,11489000000.0,9629000000.0,7671000000.0
Common Stock Equity,206223000000.0,166542000000.0,141988000000.0,118304000000.0


### YAHOOFINANCIALS

In [ ]:
from yahoofinancials import YahooFinancials
tech_stocks = ['AAPL']

In [ ]:
yahoo_financials_tech = YahooFinancials('tech_stocks')
yahoo_financials_tech

In [ ]:
tech_cash_flow_data_an = yahoo_financials_tech.get_financial_stmts('annual', 'balance')

In [ ]:
tech_cash_flow_data_an

In [7]:
yahoo_financials_tech.get_net_income()

TypeError: unsupported operand type(s) for -: 'datetime.datetime' and 'str'

In [8]:
tech_cash_flow_data_an

NameError: name 'tech_cash_flow_data_an' is not defined

### WORLD BANK

In [12]:
import pandas as pd
import world_bank_data as wb
pd.set_option('display.max_rows', 100)

ModuleNotFoundError: No module named 'world_bank_data'

In [ ]:
wb.get_topics()

In [ ]:
wb.get_sources()

In [ ]:
wb.get_countries()

In [ ]:
box = wb.get_indicators(topic=3, source=25)

In [ ]:
box

In [ ]:
wb.get_series('NY.GDP.MKTP.KD.ZG', mrv=1) 

In [ ]:
wb.get_series('SP.POP.TOTL', mrv=1)

In [ ]:
import wbdata

In [ ]:
data = wbdata.get_data('SP.POP.TOTL', country = 'SWE' )

In [ ]:
data

In [ ]:
countries = 'SWE'                                                                                             

indicators = {"IC.BUS.EASE.XQ": "doing_business", "NY.GDP.PCAP.PP.KD": "gdppc"}         

df = wbdata.get_dataframe(indicators, country=countries, convert_date=True)   

df.describe() 
df

In [ ]:
import wbdata
import pandas
import matplotlib.pyplot as plt
 
#set up the countries I want
countries = ["CL","UY","HU"]
 
#set up the indicator I want (just build up the dict if you want more than one)
indicators = {'NY.GNP.PCAP.CD':'GNI per Capita'}
 
#grab indicators above for countires above and load into data frame
df = wbdata.get_dataframe(indicators, country=countries, convert_date=False)

#df is "pivoted", pandas' unstack fucntion helps reshape it into something plottable
dfu = df.unstack(level=0)

# a simple matplotlib plot with legend, labels and a title
dfu.plot(); 
plt.legend(loc='best'); 
plt.title("GNI Per Capita ($USD, Atlas Method)"); 
plt.xlabel('Date'); plt.ylabel('GNI Per Capita ($USD, Atlas Method')

### EUROSTAT

In [13]:
import eurostat
toc = eurostat.get_toc()
toc[0]

ModuleNotFoundError: No module named 'eurostat'

In [ ]:
toc[2]

In [ ]:
import eurostat
toc_df = eurostat.get_toc_df()
toc_df

In [ ]:
toc_df.iloc[:50]

In [ ]:
box = eurostat.get_data('MED_PS43')

In [ ]:
par_values = eurostat.get_par_values('GOV_10DD_SLGD', 'geo')
par_values

In [ ]:
pars = eurostat.get_pars('GOV_10DD_SLGD')
pars

In [ ]:
data = eurostat.get_data_df('GOV_10DD_SLGD', filter_pars={ 'geo': ['AT','BE' ]})
data